# Hypothetical Document Embeddings (HyDE) Zero-Shot Dense Retrieval

## 1. Problem Formulation & Geometric Intuition
Standard dense vector retrieval encodes the user query $q$ and compares it to stored document passages $d$:

$$\operatorname{Sim}(q, d) = \cos(\mathbf{e}_q, \mathbf{e}_d)$$

However, questions and answers reside in **different semantic subspaces**:
- Queries are short, interrogative, and omit specific domain terminology (e.g., *"Why did transformer attention crash on GPU?"*).
- Target documents are long, declarative, and technical (e.g., *"CUDA Out-of-Memory exception occurs when intermediate activation tensor exceeds VRAM allocations during scaled dot-product computation..."*).

### The HyDE Paradigm
HyDE (Gao et al., 2022) bridges this semantic modality gap:
1. Pass query $q$ to an instruct LLM: *"Write a technical passage answering: {q}"*.
2. The LLM generates a **hypothetical answer document** $\hat{d}$.
3. Even if $\hat{d}$ contains hallucinated facts, its **grammatical structure, vocabulary, and embedding geometry** match the target document collection $\mathcal{D}$!
4. The embedding of the hypothetical document $\mathbf{e}_{\hat{d}}$ is used to search the vector database:

$$\operatorname{Sim}_{\text{HyDE}}(\hat{d}, d) = \cos(\mathbf{e}_{\hat{d}}, \mathbf{e}_d)$$

In [ ]:
import numpy as np

class SimulatedHyDERetriever:
    """
    Demonstrates Hypothetical Document Embeddings (HyDE) mechanics and embedding projection.
    """
    def __init__(self):
        # Simulated corpus of technical answers
        self.corpus = [
            "Gradient clipping limits maximum norm of gradients during backpropagation to prevent exploding gradients.",
            "Batch normalization normalizes layer inputs across batch dimension to stabilize training dynamics.",
            "PagedAttention eliminates memory fragmentation by storing KV cache in non-contiguous physical memory blocks.",
            "Residual connections allow gradients to flow directly through skip connections without attenuation."
        ]
        # Vocabulary mapping for synthetic embedding simulation
        self.vocab = ["gradient", "clipping", "exploding", "norm", "paged", "kv", "cache", "memory", "fragmentation"]

    def embed_text(self, text):
        """Creates term-frequency embedding normalized to unit sphere."""
        words = text.lower().replace("-", " ").split()
        vec = np.zeros(len(self.vocab), dtype=np.float32)
        for w in words:
            if w in self.vocab:
                vec[self.vocab.index(w)] += 1.0
        norm = np.linalg.norm(vec)
        return vec / (norm + 1e-9)

    def simulate_llm_hypothetical_doc(self, raw_query):
        """Simulates LLM hallucinating/generating an informative passage for query."""
        if "out of memory" in raw_query or "kv" in raw_query or "vram" in raw_query:
            return "Paged attention solves GPU memory fragmentation in KV cache by allocating non contiguous pages."
        elif "exploding" in raw_query or "gradient" in raw_query:
            return "Gradient clipping prevents exploding gradients by capping gradient norm threshold."
        return raw_query

    def search(self, query_emb, top_k=1):
        scores = []
        for idx, doc in enumerate(self.corpus):
            doc_emb = self.embed_text(doc)
            cos_sim = float(np.dot(query_emb, doc_emb))
            scores.append((cos_sim, doc))
        scores.sort(key=lambda x: x[0], reverse=True)
        return scores[:top_k]

# Comparison: Raw Query vs HyDE
engine = SimulatedHyDERetriever()
raw_query = "How to stop VRAM fragmentation in transformers?"

# Standard Search
raw_emb = engine.embed_text(raw_query)
res_standard = engine.search(raw_emb)

# HyDE Search
hypo_doc = engine.simulate_llm_hypothetical_doc(raw_query)
hyde_emb = engine.embed_text(hypo_doc)
res_hyde = engine.search(hyde_emb)

print(f"Raw Query: '{raw_query}'")
print(f"Standard Match (Sim: {res_standard[0][0]:.3f}): '{res_standard[0][1]}'")
print(f"
Generated Hypothetical Document: '{hypo_doc}'")
print(f"HyDE Match    (Sim: {res_hyde[0][0]:.3f}): '{res_hyde[0][1]}'")

## 2. Failure Modes & Edge Case Analysis

While HyDE excels in zero-shot search on broad queries, it has distinct failure modes:
1. **Hallucination Cascades**: For niche or proprietary enterprise data (e.g. *"What is Project Apollo's internal deployment endpoint?"*), the LLM will hallucinate fictitious hostnames and tokens that retrieve completely irrelevant documents.
2. **Latency Doubling**: HyDE requires an autoregressive LLM completion call prior to vector search, adding 200ms - 1500ms of end-to-end latency.

## 3. High-Yield Interview Questions & Best Practices

### Q1: When should you NOT use HyDE in a production RAG system?
**Answer:**
1. **Low-Latency Requirements**: Real-time user interfaces where P99 latency must stay under 300ms cannot afford an extra LLM generation step before vector retrieval.
2. **Proprietary Private Knowledge Bases**: When questions concern private enterprise entities or internal codebases that the public LLM was never trained on.
3. **Specific Entity Lookups**: Exact searches (e.g. part numbers, error codes like `ERR_403_ACCESS_TOKEN_EXPIRED`) where BM25 keyword search is strictly superior.

### Q2: How can multiple hypothetical documents be combined to improve recall?
**Answer:** You can sample $N$ hypothetical documents (e.g., $N=4$ at temperature $T=0.7$), embed each passage $\mathbf{e}_{\hat{d}_i}$, and compute their centroid:
$$\mathbf{e}_{\text{ensemble}} = \frac{1}{N} \sum_{i=1}^N \mathbf{e}_{\hat{d}_i}$$
Normalizing $\mathbf{e}_{\text{ensemble}}$ to unit length averages out hallucinations from any single generation while amplifying genuine semantic concepts common across samples.